[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/divyamohan1993/dip-practical/blob/main/Practical_8.ipynb)

> **Run in Google Colab:** Click the badge above to open this notebook in Google Colab. The dataset downloads automatically — no setup needed.

<style>
/* =====================================================================
   Practical Handbook — Uniform Print Formatting (CSU2543 Digital Image Processing)
   Sections are color-coded: Aim (blue), Theory (purple), Code (green),
   Output (amber), Analysis (maroon). Code blocks are uniform across all
   practicals. Page layout: 1-inch margins, justified text, Times New Roman.
   ===================================================================== */
@page { margin: 1in; }
@media print {
  body, .jp-Notebook, .jupyter-renderer { background: white !important; }
  .jp-CodeMirrorEditor, .CodeMirror { font-size: 10pt !important; }
  .jp-Cell-inputCollapser, .jp-Cell-outputCollapser, .jp-Toolbar { display: none !important; }
  .dip-section-marker { page-break-inside: avoid; }
}

.jp-RenderedHTMLCommon, .jp-RenderedMarkdown {
  font-family: 'Times New Roman', Georgia, serif;
  font-size: 12pt;
  line-height: 1.55;
  text-align: justify;
}

/* Section markers — colored heading bars */
.dip-section-marker {
  display: block;
  font-weight: bold;
  font-size: 1.35em;
  padding: 0.45em 0.7em;
  margin: 1.1em 0 0.6em 0;
  border-left: 6px solid;
  border-radius: 3px;
  letter-spacing: 0.02em;
  page-break-after: avoid;
}
.dip-section-aim      { color: #003c8f; background: #e3f2fd; border-color: #1565c0; }
.dip-section-theory   { color: #4a148c; background: #f3e5f5; border-color: #6a1b9a; }
.dip-section-code     { color: #1b5e20; background: #e8f5e9; border-color: #2e7d32; }
.dip-section-output   { color: #e65100; background: #fff3e0; border-color: #ef6c00; }
.dip-section-analysis { color: #b71c1c; background: #ffebee; border-color: #c62828; }

/* Sub-section headings within a section ("Part 1", "Part 2", ...) */
.dip-subsection {
  font-weight: 600;
  font-size: 1.1em;
  margin: 0.9em 0 0.4em 0;
  color: #2e7d32;
  border-bottom: 1px solid #c8e6c9;
  padding-bottom: 0.15em;
  page-break-after: avoid;
}

/* Code cells — uniform JetBrains Mono / Source Code Pro across all practicals */
.jp-CodeMirrorEditor, .jp-CodeCell .jp-InputArea-editor,
.CodeMirror, pre, .highlight, code {
  font-family: 'JetBrains Mono', 'Source Code Pro', Consolas, 'Courier New', monospace !important;
}
.jp-CodeCell .jp-InputArea-editor, .CodeMirror, pre {
  font-size: 10pt !important;
  background: #f6f8fa !important;
  border-left: 3px solid #2e7d32 !important;
  border-radius: 0 !important;
  padding: 8px 12px !important;
}
code { background: #f6f8fa; padding: 1px 5px; border-radius: 2px; font-size: 0.95em; }

/* Output cells — amber tint, matching the Output section colour */
.jp-OutputArea-output {
  border-left: 3px solid #ef6c00 !important;
  background: #fffaf3 !important;
  padding: 6px 10px !important;
}

/* Analysis question lists */
.dip-analysis-list { padding-left: 1.4em; }
.dip-analysis-list li { margin-bottom: 0.5em; text-align: justify; }
</style>

# Practical 8: Spatial Filtering — Smoothing and Sharpening

<span class="dip-section-marker dip-section-aim">1. Aim</span>

To implement and analyze the four canonical spatial-domain filters of digital image processing: the box (averaging) filter and median filter for noise smoothing, and the Laplacian and Sobel operators for sharpening and edge detection.

<span class="dip-section-marker dip-section-theory">2. Description / Theory</span>

**Box (mean) filter — linear, smoothing:** $\hat f(x,y) = \frac{1}{mn}\sum_{(s,t)\in S_{xy}} f(s,t)$.

**Median filter — non-linear, order-statistic:** $\hat f(x,y) = \mathrm{median}\{f(s,t):(s,t)\in S_{xy}\}$. Because the median is robust to extreme outliers, it removes salt-and-pepper noise without blurring step edges.

**Laplacian — second derivative, sharpening:**

$$ \nabla^{2}f \approx f(x{+}1,y) + f(x{-}1,y) + f(x,y{+}1) + f(x,y{-}1) - 4 f(x,y); \qquad g = f - \nabla^{2}f. $$

**Sobel — first derivative, edge detection:** orthogonal kernels $G_x, G_y$ produce the gradient magnitude $|G|\approx |G_x|+|G_y|$ and direction $\theta=\arctan(G_y/G_x)$.

<span class="dip-section-marker dip-section-code">3. Code</span>

## Code

### Setup
Import dependencies and auto-download the Gonzalez & Woods Chapter 3 dataset (test pattern, salt-and-pepper circuit board, blurry moon, contact-lens — the canonical figures for this practical).

In [ ]:
# Install dependencies (uncomment for Google Colab)
# !pip install opencv-python-headless matplotlib numpy scipy

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# === AUTO-DOWNLOAD DATASET (works in Google Colab and locally) ===
import os, urllib.request, zipfile

CHAPTER = "CH03"  # Chapter 3: Spatial Filtering
DATASET_PATH = f"datasets/{CHAPTER}/"
DOWNLOAD_BASE = "https://www.imageprocessingplace.com/downloads_V3/dip3e_downloads/dip3e_book_images"

if not os.path.exists(DATASET_PATH) or not any(f.endswith('.tif') for f in os.listdir(DATASET_PATH)):
    zip_name = f"DIP3E_{CHAPTER}_Original_Images.zip"
    url = f"{DOWNLOAD_BASE}/{zip_name}"
    print(f"Downloading {CHAPTER} dataset from imageprocessingplace.com...")
    urllib.request.urlretrieve(url, "chapter.zip")
    os.makedirs(DATASET_PATH, exist_ok=True)
    with zipfile.ZipFile("chapter.zip", "r") as z:
        for f in z.namelist():
            if f.lower().endswith(".tif"):
                fname = os.path.basename(f)
                if fname:
                    with z.open(f) as src, open(os.path.join(DATASET_PATH, fname), "wb") as dst:
                        dst.write(src.read())
    os.remove("chapter.zip")
    print(f"Downloaded {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")
else:
    print(f"Dataset ready: {len([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])} images")

images = sorted([f for f in os.listdir(DATASET_PATH) if f.endswith('.tif')])
print(f"\nAvailable images ({len(images)}):")
for i, name in enumerate(images, 1):
    print(f"  {i:2d}. {name}")

### Part 1: Box Filter (Manual Implementation)
The box filter replaces every pixel with the mean of its $k\times k$ neighbourhood. We apply it at $k = 3, 5, 9, 15, 35$ on Fig. 3.33(a) — the standard test pattern from DIP3E — to show how blurring scales with kernel size.

In [ ]:
def box_filter(img, k=3):
    """Manual box filter via 2D correlation with a uniform kernel."""
    p = k // 2
    kernel = np.ones((k, k), dtype=np.float64) / (k * k)
    padded = np.pad(img.astype(np.float64), p, mode='edge')
    out = np.zeros_like(img, dtype=np.float64)
    h, w = img.shape
    for i in range(h):
        for j in range(w):
            out[i, j] = np.sum(padded[i:i+k, j:j+k] * kernel)
    return np.clip(out, 0, 255).astype(np.uint8)

test_pattern_file = "Fig0333(a)(test_pattern_blurring_orig).tif"
tp = cv2.imread(os.path.join(DATASET_PATH, test_pattern_file), cv2.IMREAD_GRAYSCALE)
print(f"Test pattern: {test_pattern_file}, shape={tp.shape}")

kernel_sizes = [3, 5, 9, 15, 35]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()
axes[0].imshow(tp, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original\nFig0333(a)'); axes[0].axis('off')
for idx, k in enumerate(kernel_sizes, start=1):
    out = box_filter(tp, k=k)
    axes[idx].imshow(out, cmap='gray', vmin=0, vmax=255)
    axes[idx].set_title(f'Box {k}x{k}'); axes[idx].axis('off')
plt.suptitle('Box Filter: Effect of Kernel Size on Test Pattern', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Part 2: Median Filter on Salt-and-Pepper Noise
Apply the manual median filter to Fig. 3.35(a), the circuit-board image corrupted with 5% salt-and-pepper impulse noise. Median filtering removes the impulses cleanly because they are extreme values that fall outside any reasonable rank position in a sorted neighbourhood.

In [ ]:
def median_filter(img, k=3):
    """Manual median filter via sliding window."""
    p = k // 2
    padded = np.pad(img, p, mode='edge')
    out = np.zeros_like(img)
    h, w = img.shape
    for i in range(h):
        for j in range(w):
            out[i, j] = np.median(padded[i:i+k, j:j+k])
    return out.astype(np.uint8)

salt_pepper_file = "Fig0335(a)(ckt_board_saltpep_prob_pt05).tif"
if salt_pepper_file not in images:
    # Some distributions name it differently
    candidates = [f for f in images if 'ckt' in f.lower() and ('saltpep' in f.lower() or 'saltpr' in f.lower())]
    salt_pepper_file = candidates[0] if candidates else images[0]
    print(f"Substituted: {salt_pepper_file}")

noisy = cv2.imread(os.path.join(DATASET_PATH, salt_pepper_file), cv2.IMREAD_GRAYSCALE)
print(f"Noisy image: {salt_pepper_file}, shape={noisy.shape}")

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
axes[0].imshow(noisy, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original\n(Salt & Pepper, p=0.05)'); axes[0].axis('off')
for idx, k in enumerate([3, 5, 7, 9], start=1):
    out = median_filter(noisy, k=k)
    axes[idx].imshow(out, cmap='gray', vmin=0, vmax=255)
    axes[idx].set_title(f'Median {k}x{k}'); axes[idx].axis('off')
plt.suptitle('Median Filter: Removing Impulse Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Part 3: Box vs Median — Edge Preservation
Apply both filters at the same kernel size to the same noisy input and view side-by-side. The box filter blurs edges along with the noise; the median filter removes the noise but leaves edges intact, which is the practical reason it is preferred for impulse noise.

In [ ]:
k = 3
box_out = box_filter(noisy, k=k)
med_out = median_filter(noisy, k=k)
removed_box = np.abs(noisy.astype(int) - box_out.astype(int)).astype(np.uint8)
removed_med = np.abs(noisy.astype(int) - med_out.astype(int)).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(noisy,   cmap='gray', vmin=0, vmax=255); axes[0, 0].set_title('Noisy input'); axes[0, 0].axis('off')
axes[0, 1].imshow(box_out, cmap='gray', vmin=0, vmax=255); axes[0, 1].set_title(f'Box {k}x{k}'); axes[0, 1].axis('off')
axes[0, 2].imshow(med_out, cmap='gray', vmin=0, vmax=255); axes[0, 2].set_title(f'Median {k}x{k}'); axes[0, 2].axis('off')
axes[1, 0].axis('off')
axes[1, 1].imshow(removed_box, cmap='hot'); axes[1, 1].set_title('Removed by Box (noise + edges)'); axes[1, 1].axis('off')
axes[1, 2].imshow(removed_med, cmap='hot'); axes[1, 2].set_title('Removed by Median (mostly noise)'); axes[1, 2].axis('off')
plt.suptitle('Box vs Median: Edge Preservation Under Impulse Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean energy removed by box   = {removed_box.mean():.2f}")
print(f"Mean energy removed by median = {removed_med.mean():.2f}")

### Part 4: Laplacian Sharpening
Compute $\nabla^{2}f$ with the standard 4-connected and 8-connected Laplacian kernels, and form the sharpened image $g = f - \nabla^{2}f$ (the centre coefficient of our kernel is positive +4, so we **subtract** the response). Demonstrated on Fig. 3.38(a), the blurry image of the moon.

In [ ]:
lap4 = np.array([[ 0, -1,  0],
                 [-1,  4, -1],
                 [ 0, -1,  0]], dtype=np.float64)

lap8 = np.array([[-1, -1, -1],
                 [-1,  8, -1],
                 [-1, -1, -1]], dtype=np.float64)

moon_file = "Fig0338(a)(blurry_moon).tif"
if moon_file not in images:
    candidates = [f for f in images if 'moon' in f.lower()]
    moon_file = candidates[0] if candidates else images[0]
    print(f"Substituted: {moon_file}")

moon = cv2.imread(os.path.join(DATASET_PATH, moon_file), cv2.IMREAD_GRAYSCALE)
print(f"Moon image: {moon_file}, shape={moon.shape}")

lap4_resp = cv2.filter2D(moon.astype(np.float64), ddepth=-1, kernel=lap4)
lap8_resp = cv2.filter2D(moon.astype(np.float64), ddepth=-1, kernel=lap8)

sharp4 = np.clip(moon.astype(np.float64) - lap4_resp, 0, 255).astype(np.uint8)
sharp8 = np.clip(moon.astype(np.float64) - lap8_resp, 0, 255).astype(np.uint8)

def to_disp(a):
    a = a - a.min()
    return (255.0 * a / max(a.max(), 1e-9)).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(moon, cmap='gray', vmin=0, vmax=255); axes[0, 0].set_title('Original (blurry moon)'); axes[0, 0].axis('off')
axes[0, 1].imshow(to_disp(lap4_resp), cmap='gray'); axes[0, 1].set_title('Laplacian response (4-neighbour)'); axes[0, 1].axis('off')
axes[0, 2].imshow(sharp4, cmap='gray', vmin=0, vmax=255); axes[0, 2].set_title('Sharpened: f - lap4(f)'); axes[0, 2].axis('off')
axes[1, 0].imshow(moon, cmap='gray', vmin=0, vmax=255); axes[1, 0].set_title('Original (blurry moon)'); axes[1, 0].axis('off')
axes[1, 1].imshow(to_disp(lap8_resp), cmap='gray'); axes[1, 1].set_title('Laplacian response (8-neighbour)'); axes[1, 1].axis('off')
axes[1, 2].imshow(sharp8, cmap='gray', vmin=0, vmax=255); axes[1, 2].set_title('Sharpened: f - lap8(f)'); axes[1, 2].axis('off')
plt.suptitle('Laplacian Sharpening: 4-neighbour vs 8-neighbour kernel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Part 5: Sobel Edge Detection
Compute the horizontal and vertical Sobel responses $G_x$ and $G_y$, the gradient magnitude $|G|\approx |G_x|+|G_y|$, and the gradient direction $\theta=\arctan(G_y/G_x)$, on the contact-lens image (Fig. 3.42(a)).

In [ ]:
sx = np.array([[-1, 0, 1],
               [-2, 0, 2],
               [-1, 0, 1]], dtype=np.float64)
sy = np.array([[-1, -2, -1],
               [ 0,  0,  0],
               [ 1,  2,  1]], dtype=np.float64)

lens_file = "Fig0342(a)(contact_lens_original).tif"
if lens_file not in images:
    candidates = [f for f in images if 'lens' in f.lower() or 'contact' in f.lower()]
    lens_file = candidates[0] if candidates else images[0]
    print(f"Substituted: {lens_file}")

lens = cv2.imread(os.path.join(DATASET_PATH, lens_file), cv2.IMREAD_GRAYSCALE)
print(f"Edge-detection image: {lens_file}, shape={lens.shape}")

Gx = cv2.filter2D(lens.astype(np.float64), ddepth=-1, kernel=sx)
Gy = cv2.filter2D(lens.astype(np.float64), ddepth=-1, kernel=sy)
Gmag = np.abs(Gx) + np.abs(Gy)
Gmag_disp = np.clip(255.0 * Gmag / max(Gmag.max(), 1e-9), 0, 255).astype(np.uint8)
Gtheta = np.arctan2(Gy, Gx)  # radians, range [-pi, pi]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(lens, cmap='gray', vmin=0, vmax=255); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
axes[0, 1].imshow(np.abs(Gx), cmap='gray'); axes[0, 1].set_title('|Gx| (vertical edges)'); axes[0, 1].axis('off')
axes[0, 2].imshow(np.abs(Gy), cmap='gray'); axes[0, 2].set_title('|Gy| (horizontal edges)'); axes[0, 2].axis('off')
axes[1, 0].imshow(Gmag_disp, cmap='gray'); axes[1, 0].set_title('|G| = |Gx|+|Gy|'); axes[1, 0].axis('off')
th_disp = ((Gtheta + np.pi) / (2 * np.pi) * 255).astype(np.uint8)
axes[1, 1].imshow(th_disp, cmap='hsv'); axes[1, 1].set_title('Gradient direction \u03B8'); axes[1, 1].axis('off')
thresh = (Gmag_disp > 64).astype(np.uint8) * 255
axes[1, 2].imshow(thresh, cmap='gray', vmin=0, vmax=255); axes[1, 2].set_title('Thresholded edges'); axes[1, 2].axis('off')
plt.suptitle('Sobel Edge Detection: Gx, Gy, magnitude, direction, threshold', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Output

All output (printed values, matrix prints, and rendered figures) appears immediately below the corresponding code cells when the notebook is executed top-to-bottom.

<span class="dip-section-marker dip-section-output">4. Output</span>

*All output (printed values, computed statistics, and rendered figures) appears immediately below the corresponding code cells when this notebook is executed top-to-bottom.*

<span class="dip-section-marker dip-section-analysis">5. Analysis / Conclusion</span>

**1. As the box-filter kernel grows from $3\times3$ to $35\times35$, what happens to the image, and why?**  
Each output pixel is averaged over an $k^{2}$ neighbourhood, so the bandwidth of the smoothing kernel shrinks as $1/k$ in the spatial-frequency sense. Fine textures and small features (smaller than the kernel) are attenuated first; at $k=15$ medium structure is gone; at $k=35$ even the large geometric blocks of the test pattern lose their sharp boundaries. Box filtering is a low-pass operation whose cut-off frequency is inversely proportional to kernel size.

**2. Why does the median filter remove salt-and-pepper noise without blurring edges, while the box filter blurs both noise and edges?**  
Salt-and-pepper noise consists of pixels at the extremes (0 or 255). The box filter averages these extreme values into the result, which both spreads the noise across neighbours and softens any genuine edge that lies in the same window. The median is an order statistic: as long as fewer than half the pixels in the window are corrupted, the median is drawn from the clean pixels and the impulse is discarded entirely. A step edge inside the window has the majority of its values on one side, so the median sits firmly on that side and the edge is preserved bit-exact.

**3. The Laplacian is a second-derivative operator. Why does subtracting its response sharpen the image instead of darkening it?**  
At a bright peak, $\nabla^{2}f$ is negative (the centre is brighter than the average of its neighbours), so $f - \nabla^{2}f$ adds to the peak — making it brighter. At a dark valley the response is positive, so the subtraction makes the valley darker. The net effect is to amplify local contrast around every edge. The 8-neighbour kernel includes diagonal neighbours and therefore responds to edges of every orientation more uniformly — it produces stronger, slightly noisier sharpening than the 4-neighbour kernel.

**4. Sobel uses two separate kernels and combines their magnitudes — why not use the Laplacian for edge detection instead?**  
The Laplacian is isotropic and gives a single scalar response per pixel; it reacts strongly to noise (second derivatives amplify high frequencies) and provides no direction information. Sobel computes first derivatives along two orthogonal axes; it is less noise-sensitive (first derivatives + a built-in smoothing factor of $[1\,2\,1]$ along the perpendicular axis), and it yields both magnitude and direction, which downstream stages such as non-maximum suppression and Canny edge linking require. Laplacian is preferred for sharpening; Sobel is preferred for *detecting and tracing* edges.